In [ ]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr#, boxscoreusagev2 as usg
import pandas as pd
import random
from time import sleep

Fetch Games

In [ ]:
# Fetch all game IDs since 2015
import pandas as pd
seasons = range(2015, 2025)
games = pd.DataFrame()
gamelog = pd.DataFrame()

for season in seasons:
    gamelog = log.LeagueGameLog(season=season).get_data_frames()[0][['GAME_ID', 'GAME_DATE', 'TEAM_NAME', 'MATCHUP', 'WL']]
    games = pd.concat([games, gamelog])


In [19]:
import time
from functools import wraps
from requests.exceptions import ReadTimeout, ConnectionError

def retry(retries=3, delay=10):
    """
    Decorator to retry API calls with a specified number of retries and delay between attempts.
    
    Args:
        retries (int): Number of retry attempts
        delay (int): Delay between retries in seconds
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < retries:
                try:
                    return func(*args, **kwargs)
                except (ReadTimeout, ConnectionError) as e:
                    print(f"Attempt {attempts + 1} failed: {str(e)}")
                    if attempts < retries - 1:
                        print(f"Waiting {delay} seconds before retrying...")
                        time.sleep(delay)
                    attempts += 1
            raise Exception(f"Failed after {retries} attempts")
        return wrapper
    return decorator

In [112]:
scr.BoxScoreScoringV2(game_id='0021500003').get_data_frames()[1]

,GAME_ID,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,TEAM_CITY,MIN,PCT_FGA_2PT,PCT_FGA_3PT,PCT_PTS_2PT,PCT_PTS_2PT_MR,...,PCT_PTS_FB,PCT_PTS_FT,PCT_PTS_OFF_TOV,PCT_PTS_PAINT,PCT_AST_2PM,PCT_UAST_2PM,PCT_AST_3PM,PCT_UAST_3PM,PCT_AST_FGM,PCT_UAST_FGM
0,0021500003,1610612740,Pelicans,NOP,New Orleans,240.000000:00,0.783,0.217,0.611,0.189,...,0.189,0.20,0.305,0.421,0.586,0.414,0.667,0.333,0.600,0.400
1,0021500003,1610612744,Warriors,GSW,Golden State,240.000000:00,0.688,0.313,0.577,0.090,...,0.117,0.18,0.234,0.486,0.688,0.313,0.778,0.222,0.707,0.293


In [138]:
games[(games.GAME_ID=='0021501190') & (games.TEAM_NAME=='Golden State Warriors')]

AttributeError: 'DataFrame' object has no attribute 'GAME_ID'

In [29]:
from nba_api.stats.endpoints import (
    leaguegamelog as log,
    boxscoretraditionalv2 as trd,
    boxscoreadvancedv2 as adv,
    boxscorefourfactorsv2 as ffs,
    boxscoremiscv2 as msc,
    boxscoreplayertrackv2 as trk,
    boxscorescoringv2 as scr
)
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from retry_decorator import retry
import logging
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class NBADataProcessor:
    def __init__(self, seasons: List[str]):
        self.seasons = seasons
        self.endpoints = {
            'traditional': trd.BoxScoreTraditionalV2,
            'advanced': adv.BoxScoreAdvancedV2,
            'four_factors': ffs.BoxScoreFourFactorsV2,
            'misc': msc.BoxScoreMiscV2,
            'tracking': trk.BoxScorePlayerTrackV2
        }

    @retry(retries=3, delay=10)
    def fetch_game_ids(self, season: str) -> List[str]:
        """Fetch game IDs for a given season with retry mechanism."""
        try:
            gamelog = log.LeagueGameLog(season=season)
            games = gamelog.get_data_frames()[0]
            return games['GAME_ID'].unique().tolist()
        except Exception as e:
            logger.error(f"Error fetching game IDs for season {season}: {str(e)}")
            raise

    @retry(retries=3, delay=10)
    def fetch_game_data(self, game_id: str) -> Dict[str, pd.DataFrame]:
        """Fetch all boxscore data for a single game with retry mechanism."""
        game_data = {}
        for key, endpoint in self.endpoints.items():
            try:
                bs = endpoint(game_id=game_id)
                team_stats = bs.get_data_frames()[1]  # Team stats DataFrame
                game_data[key] = team_stats
                time.sleep(1)  # Rate limiting
            except Exception as e:
                logger.error(f"Error fetching {key} data for game {game_id}: {str(e)}")
                raise
        return game_data

    def adjust_to_pace_100(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adjust statistics to per-100 possessions."""
        pace_cols = ['PTS', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 
                    'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF']
        
        df_adjusted = df.copy()
        pace_factor = 100 / df['PACE']
        
        for col in pace_cols:
            if col in df.columns:
                df_adjusted[col] = df[col] * pace_factor
        
        return df_adjusted

    def calculate_matchup_differences(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate differences between teams in the same game."""
        diff_cols = ['PTS', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 
                    'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF']
        
        result_dfs = []
        for game_id in df['GAME_ID'].unique():
            game_df = df[df['GAME_ID'] == game_id].copy()
            if len(game_df) == 2:  # Ensure we have both teams
                team1, team2 = game_df.iloc[0], game_df.iloc[1]
                for col in diff_cols:
                    if col in df.columns:
                        game_df[f'{col}_DIFF'] = game_df[col] - game_df[col].iloc[::-1].values
                result_dfs.append(game_df)
        
        return pd.concat(result_dfs, ignore_index=True)

    def process_data(self) -> pd.DataFrame:
        """Main processing function to fetch and transform all data."""
        all_game_data = []
        
        for season in self.seasons:
            logger.info(f"Processing season: {season}")
            try:
                game_ids = self.fetch_game_ids(season)
                
                for game_id in game_ids:
                    #logger.info(f"Processing game: {game_id}")
                    try:
                        # Fetch and combine all boxscore data
                        game_data = self.fetch_game_data(game_id)
                        
                        # Merge all boxscore types
                        merged = pd.concat(
                            [df.set_index('TEAM_ID') for df in game_data.values()],
                            axis=1
                        ).reset_index()
                        
                        # Remove duplicate columns
                        merged = merged.loc[:, ~merged.columns.duplicated()]
                        merged.GAME_ID = game_id
                        
                        all_game_data.append(merged)
                        
                    except Exception as e:
                        logger.error(f"Error processing game {game_id}: {str(e)}")
                        continue
                    
            except Exception as e:
                logger.error(f"Error processing season {season}: {str(e)}")
                continue

        # Combine all data
        if not all_game_data:
            raise ValueError("No data was successfully processed")
        
        final_df = pd.concat(all_game_data, ignore_index=True)
        
        # Adjust to per-100 possessions
        final_df = self.adjust_to_pace_100(final_df)
        
        # Calculate matchup differences
        final_df = self.calculate_matchup_differences(final_df)
        
        # Select and reorder columns
        essential_cols = ['GAME_ID', 'TEAM_NAME', 'MATCHUP', 'GAME_DATE']
        stat_cols = [col for col in final_df.columns if col not in essential_cols]
        final_df = final_df[essential_cols + stat_cols]
        
        return final_df

def main():
    # Example usage
    #seasons = ['2021-22', '2022-23']  # Add your seasons here
    seasons = range(2015,2025)
    processor = NBADataProcessor(seasons)
    
    try:
        df = processor.process_data()
        # Save to CSV or perform additional analysis
        df.to_csv('nba_processed_data.csv', index=False)
        #logger.info("Data processing completed successfully")
    except Exception as e:
        logger.error(f"Processing failed: {str(e)}")

if __name__ == "__main__":
    main()

INFO:__main__:Processing season: 2015


KeyboardInterrupt: 

In [158]:
games_2025 = log.LeagueGameLog(season='2024').get_data_frames()[0][['GAME_ID', 'GAME_DATE', 'TEAM_NAME', 'MATCHUP', 'WL']]

In [ ]:
merged_games

,GAME_ID,GAME_DATE,TEAM_NAME,MATCHUP,WL
0,0022400062,2024-10-22,Los Angeles Lakers,LAL vs. MIN,W
1,0022400061,2024-10-22,New York Knicks,NYK @ BOS,L
2,0022400062,2024-10-22,Minnesota Timberwolves,MIN @ LAL,L
3,0022400061,2024-10-22,Boston Celtics,BOS vs. NYK,W
4,0022400066,2024-10-23,Milwaukee Bucks,MIL @ PHI,W
...,...,...,...,...,...
1333,0022400644,2025-01-25,Golden State Warriors,GSW vs. LAL,None
1334,0022400644,2025-01-25,Los Angeles Lakers,LAL @ GSW,None
1335,0022400639,2025-01-25,Toronto Raptors,TOR @ ATL,None
1336,0022400639,2025-01-25,Atlanta Hawks,ATL vs. TOR,None


In [6]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr#, boxscoreusagev2 as usg
games_2025 = log.LeagueGameLog(season='2024').get_data_frames()[0]

In [ ]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr#, boxscoreusagev2 as usg
import pandas as pd
from retry_decorator import retry

#@retry(retries=3, delay=10)
def fetch_game_data(game_id):
    trd_temp = trd.BoxScoreTraditionalV2(game_id=game_id, timeout=3).get_data_frames()[1]
    return trd_temp

trd_stats = pd.DataFrame()
for game in games_2025.GAME_ID:
    try:
        game_data = fetch_game_data(game)
        trd_stats = pd.concat([trd_stats, game_data], ignore_index=True)
    except Exception as e:
        print(f"Failed to fetch data for game {game}: {str(e)}")
        continue

Failed to fetch data for game 0022400084: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400083: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400083: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400081: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400093: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400087: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400094: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=3)
Failed to fetch data for game 0022400100: HTTPSConnectionPool(host='stats.nba.com', port=443): Read time

KeyboardInterrupt: 

In [18]:
trd_stats.to_csv('data/trd_stats_2025.csv')

,GAME_ID,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,TEAM_CITY,MIN,FGM,FGA,FG_PCT,FG3M,...,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS
0,0022400062,1610612750,Timberwolves,MIN,Minnesota,240.000000:00,35,85,0.412,13,...,12,35,47,17,4,1,15,22,103,-7.0
1,0022400062,1610612747,Lakers,LAL,Los Angeles,240.000000:00,42,95,0.442,5,...,15,31,46,22,7,8,7,22,110,7.0
2,0022400061,1610612752,Knicks,NYK,New York,240.000000:00,43,78,0.551,11,...,5,29,34,20,2,3,11,12,109,-23.0
3,0022400061,1610612738,Celtics,BOS,Boston,240.000000:00,48,95,0.505,29,...,11,29,40,33,6,3,3,15,132,23.0
4,0022400061,1610612752,Knicks,NYK,New York,240.000000:00,43,78,0.551,11,...,5,29,34,20,2,3,11,12,109,-23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2605,0022400532,1610612737,Hawks,ATL,Atlanta,240.000000:00,36,89,0.404,11,...,9,31,40,19,11,5,16,18,96,-4.0
2606,0022400660,1610612755,76ers,PHI,Philadelphia,240.000000:00,43,91,0.473,11,...,12,31,43,20,14,4,9,20,118,14.0
2607,0022400660,1610612747,Lakers,LAL,Los Angeles,240.000000:00,37,72,0.514,10,...,4,35,39,25,5,3,22,19,104,-14.0
2608,0022400660,1610612755,76ers,PHI,Philadelphia,240.000000:00,43,91,0.473,11,...,12,31,43,20,14,4,9,20,118,14.0
